<a href="https://colab.research.google.com/github/Svatunek-Lab/teaching/blob/main/2026_University_of_Malaya/Molecule_to_3D_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecule → 3D
Enter a **name, CAS number, or SMILES**. [Draw a molecule → copy SMILES](https://draw.cheminfo.org/).

In [ ]:
!pip -q install rdkit py3Dmol ipywidgets

In [ ]:

#@title Molecular Structure Generator { display-mode: "form" }

# Input parameters (native Colab forms)
molecule = "aspirin" #@param {type:"string"}
charge = 0 #@param {type:"integer"}
multiplicity = 1 #@param {type:"integer"}

import html
import json
import base64
import urllib.parse
import urllib.request

from IPython.display import HTML, display
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem, Draw
import py3Dmol


# --------------------------------------------------
# 1. Resolve name, CAS number or SMILES
# --------------------------------------------------

def resolve_smiles(text):
    text = text.strip()

    if not text:
        raise ValueError("Enter a molecular name, CAS number or SMILES.")

    # Try interpreting the input as SMILES without printing errors
    with rdBase.BlockLogs():
        candidate = Chem.MolFromSmiles(text)

    if candidate is not None:
        return text

    # Otherwise look up the molecule on PubChem
    encoded = urllib.parse.quote(text, safe="")
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded}/property/IsomericSMILES/JSON"
    )

    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            data = json.load(response)

        properties = data["PropertyTable"]["Properties"][0]

        for key in ("SMILES", "IsomericSMILES",
                    "ConnectivitySMILES", "CanonicalSMILES"):
            if properties.get(key):
                return properties[key]

    except Exception as exc:
        raise ValueError(
            f"Could not resolve '{text}' via PubChem. "
            "Try entering a SMILES string."
        ) from exc

    raise ValueError("No SMILES returned by PubChem.")


# --------------------------------------------------
# 2. Generate and optimize 3D coordinates
# --------------------------------------------------

def generate_3d(smiles):
    base = Chem.MolFromSmiles(smiles)

    if base is None:
        raise ValueError("Invalid SMILES.")

    mol = Chem.AddHs(base)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42

    status = AllChem.EmbedMolecule(mol, params)

    if status != 0:
        raise ValueError("3D structure generation failed.")

    method = "ETKDGv3"

    if AllChem.MMFFHasAllMoleculeParams(mol):
        result = AllChem.MMFFOptimizeMolecule(
            mol, maxIters=400
        )
        method += " + MMFF94"

    elif AllChem.UFFHasAllMoleculeParams(mol):
        result = AllChem.UFFOptimizeMolecule(
            mol, maxIters=400
        )
        method += " + UFF"

    else:
        result = None
        method += " (no force-field optimization)"

    if result == 1:
        print("Warning: geometry optimization did not fully converge.")

    return base, mol, method


# --------------------------------------------------
# 3. Create XYZ and Gaussian input files
# --------------------------------------------------

def create_files(mol, label, charge, multiplicity):
    conf = mol.GetConformer()
    lines = []

    for atom in mol.GetAtoms():
        p = conf.GetAtomPosition(atom.GetIdx())

        lines.append(
            f"{atom.GetSymbol():<2} "
            f"{p.x:13.6f} "
            f"{p.y:13.6f} "
            f"{p.z:13.6f}"
        )

    coords = "\n".join(lines)

    xyz = (
        f"{len(lines)}\n"
        f"{label}\n"
        f"{coords}\n"
    )

    # Simple Gaussian 16 placeholder calculation
    gaussian = (
        "%chk=molecule.chk\n"
        "#p HF/3-21G Opt\n\n"
        f"{label}\n\n"
        f"{charge} {multiplicity}\n"
        f"{coords}\n\n"
    )

    return xyz, gaussian


# --------------------------------------------------
# 4. Browser download links
# --------------------------------------------------

def download_link(content, filename, title):
    encoded = base64.b64encode(
        content.encode("utf-8")
    ).decode("ascii")

    return f"""
    <a href="data:text/plain;base64,{encoded}"
       download="{html.escape(filename, quote=True)}"
       style="
           display:inline-block;
           padding:10px 17px;
           margin:5px 8px 5px 0;
           background:#555;
           color:white;
           text-decoration:none;
           border-radius:5px;
           font-family:Arial,sans-serif;
           font-size:14px;
       ">
       {html.escape(title)}
    </a>
    """


# --------------------------------------------------
# 5. Build 3D viewer inside a positioned iframe
# --------------------------------------------------

def make_3d_viewer(mol, width=450, height=340):
    viewer = py3Dmol.view(
        width=width,
        height=height
    )

    viewer.addModel(
        Chem.MolToMolBlock(mol),
        "sdf"
    )

    viewer.setStyle({
        "stick": {"radius": 0.17},
        "sphere": {"scale": 0.26}
    })

    viewer.setBackgroundColor("white")
    viewer.zoomTo()

    viewer_html = viewer._make_html()

    # Isolate the viewer inside a correctly sized
    # positioning context, avoiding top-left placement.
    document = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <meta charset="utf-8">
    <style>
        html, body {{
            margin:0;
            padding:0;
            width:100%;
            height:100%;
            overflow:hidden;
        }}
        #container {{
            position:relative;
            width:{width}px;
            height:{height}px;
            overflow:hidden;
        }}
    </style>
    </head>
    <body>
        <div id="container">
            {viewer_html}
        </div>
    </body>
    </html>
    """

    return f"""
    <iframe
        srcdoc="{html.escape(document, quote=True)}"
        width="{width}"
        height="{height}"
        style="
            display:block;
            position:relative;
            border:0;
            width:{width}px;
            height:{height}px;
            overflow:hidden;
        ">
    </iframe>
    """


# --------------------------------------------------
# 6. Run and display everything
# --------------------------------------------------

try:
    if multiplicity < 1:
        raise ValueError("Multiplicity must be at least 1.")

    smiles = resolve_smiles(molecule)
    base, mol, method = generate_3d(smiles)

    xyz, gaussian = create_files(
        mol, molecule, charge, multiplicity
    )

    # Generate 2D SVG
    svg = Draw.MolsToGridImage(
        [base],
        molsPerRow=1,
        subImgSize=(390, 330),
        useSVG=True
    )

    # Generate 3D HTML
    viewer_html = make_3d_viewer(mol)

    # Download links
    downloads = (
        download_link(xyz, "molecule.xyz", "Download XYZ")
        + download_link(
            gaussian, "molecule.com", "Download COM"
        )
        + download_link(
            gaussian, "molecule.gjf", "Download GJF"
        )
    )

    # Display structures and downloads
    output_html = f"""
    <div style="font-family:Arial,sans-serif;">

        <div style="margin:8px 0;">
            <b>SMILES:</b>
            <code>{html.escape(smiles)}</code>
        </div>

        <div style="margin:4px 0 14px;color:#666;">
            3D generation: {html.escape(method)}
        </div>

        <div style="
            display:flex;
            flex-wrap:wrap;
            align-items:flex-start;
            gap:20px;
        ">

            <div style="width:400px;">
                <b>2D structure</b>
                <div style="
                    position:relative;
                    width:400px;
                    height:340px;
                    background:white;
                ">
                    {str(svg)}
                </div>
            </div>

            <div style="width:450px;">
                <b>3D structure (rotatable)</b>
                <div style="
                    position:relative;
                    width:450px;
                    height:340px;
                    overflow:hidden;
                    margin-top:5px;
                ">
                    {viewer_html}
                </div>
            </div>

        </div>

        <div style="margin:16px 0;">
            {downloads}
        </div>

        <div style="margin-top:12px;">
            <b>XYZ coordinates (Å)</b>
        </div>

        <textarea readonly
            style="
                box-sizing:border-box;
                width:100%;
                height:230px;
                margin-top:8px;
                padding:12px;
                font-family:monospace;
                font-size:13px;
                background:#f5f5f5;
                color:#222;
                border:1px solid #ddd;
                border-radius:5px;
            ">{html.escape(xyz)}</textarea>

    </div>
    """

    display(HTML(output_html))

except Exception as exc:
    print("Error:", exc)
